# 00 - Getting started with `gaknot`

This notebook introduces the package's mathematical data model and the small set of constructors from which the later tutorials are built. It is intended to be read before the invariant-specific notebooks.

## Learning objectives

By the end of the notebook you should be able to:

- run `gaknot` from a repository checkout with the SageMath kernel;
- construct torus knots, iterated torus knots, connected sums, and concordance inverses;
- read the nested structural description used throughout the package; and
- request the ordinary Levine--Tristram signature and Alexander polynomial.

All cabling pairs are written from the innermost knot to the outermost pattern. Thus `[(2, 3), (2, 5)]` means the `(2,5)`-cable of the trefoil, not the other way around.

## 1. Load the checkout

The following cell makes the notebook independent of an editable installation. It locates the repository root, inserts `src/` on Python's import path, and then imports only public package objects. The `.sage` sources must first have been preparsed with `make build`.

In [ ]:
from pathlib import Path
import sys

repository_root = Path.cwd()
if repository_root.name == "notebooks":
    repository_root = repository_root.parent

source_directory = repository_root / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

from sage.all import QQ
from sage.env import SAGE_VERSION
from gaknot import GeneralizedAlgebraicKnot

print(f"SageMath version: {SAGE_VERSION}")
print(f"gaknot source: {source_directory}")

If the import fails, check that the selected Jupyter kernel is **SageMath**, not plain Python, and run `make build` in the repository root.

## 2. Construct basic examples

`torus_knot(p, q)` creates a one-summand positive torus knot. `iterated_torus_knot(...)` accepts the entire inside-to-outside cabling sequence. The method `cable(p, q)` is a convenient equivalent when extending a one-summand knot.

In [ ]:
trefoil = GeneralizedAlgebraicKnot.torus_knot(2, 3)

cable_from_sequence = GeneralizedAlgebraicKnot.iterated_torus_knot(
    [(2, 3), (2, 5)]
)
cable_from_method = trefoil.cable(2, 5)

print("trefoil:", trefoil)
print("iterated knot:", cable_from_sequence)
print("the two cable constructions agree:", cable_from_sequence == cable_from_method)

The equality compares the stored structural descriptions. In this example the first pair `(2,3)` is the companion and the final pair `(2,5)` is the outer satellite pattern.

## 3. Connected sums, signs, and structural provenance

Addition concatenates connected-sum summands; unary minus reverses every summand sign. The package deliberately does **not** simplify `K + (-K)` to an empty description. Retaining both summands is important when later calculations attach homology coordinates or Casson--Gordon contributions to individual pieces.

In [ ]:
formal_slice_pair = cable_from_sequence + (-cable_from_sequence)
mixed_sum = cable_from_sequence + (-trefoil)

print("K + (-K):", formal_slice_pair)
print("number of visible summands:", len(formal_slice_pair))
print("mixed description:", mixed_sum.description)

The public `description` property returns a defensive copy. Editing the returned list cannot mutate the knot object or invalidate invariants already computed from it.

In [ ]:
external_description = mixed_sum.description
external_description[0][1].append((3, 7))

print("edited external copy:", external_description)
print("unchanged stored description:", mixed_sum.description)

## 4. Basic invariants

`signature()` returns a sparse periodic signature function. Its dictionary records **jump weights** rather than sampled values: a weight `j` means that the signature changes by `2*j` at that argument. `alexander_polynomial()` returns the package's normalized polynomial over `ZZ[t]`.

In [ ]:
signature = cable_from_sequence.signature()
alexander = cable_from_sequence.alexander_polynomial()

print("signature jump weights:")
print(signature.jumps_counter)
print()
print("normalized Alexander polynomial:")
print(alexander)

Arguments are elements of `R/Z`: the rational number `x` denotes the unit complex number `exp(2*pi*i*x)`. Exact Sage rationals avoid floating-point ambiguity at discontinuities.

In [ ]:
sample_arguments = [QQ(0), QQ(1) / 10, QQ(1) / 6, QQ(1) / 2]
{argument: signature(argument) for argument in sample_arguments}

The value exactly at a jump uses the midpoint convention. The next notebook develops that convention, the sparse algebra, periodicity, and plotting in detail.

## 5. What the structural model does and does not claim

- Every cabling pair must contain coprime integers greater than one, so it describes a torus **knot** rather than a torus link.
- A sign is either `+1` or `-1`; arbitrary multiplicities are represented by repeated connected-sum entries.
- The model records a chosen decomposition. It is not a canonical normal form in the knot concordance group.
- Different invariants support different subfamilies. Constructing a GA-knot does not imply that every specialized invariant is implemented for it.

Those explicit support boundaries are part of the API. Later notebooks show both successful computations and how a coverage gap is reported.

## Exercises

1. Construct `T(3,4)` and its `(2,5)`-cable. Inspect the description to confirm the order of the two pairs.
2. Form the connected sum of two trefoils and one negative trefoil. How many summands remain visible?
3. Compare the Alexander polynomials of `trefoil` and `-trefoil`. Explain why the sign does not alter this normalized invariant.